# Setting Up

In [11]:
import torch
import torch.nn.functional as F
import torch.nn as nn

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [4]:
!nvidia-smi

Tue Jul 28 09:04:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.74                 KMD Version: 610.74        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   37C    P0             26W /  160W |       0MiB /  12282MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Basic Sequential.nn

In Reinforcement Learning, the neural network acts as the agent's "Brain." For a Deep Q-Network (DQN), the goal is to take a state tensor `[Batch, Time, Features]` and output a Q-value for every possible action `[Batch, Time, Actions]`.

The fastest way to build a neural network in PyTorch is using `nn.Sequential`.

It acts as a straight pipe: data goes in layer 1, and flows sequentially to the end.

Let's build a simple 3-layer Multi-Layer Perceptron (MLP) and try to feed it a batch from our RL DataLoader.

In [5]:
# 1. Define the network dimensions
INPUT_FEATURES = 4 
HIDDEN_SIZE = 64
NUM_ACTIONS = 2

# 2. Build the Naive Model
naive_q_network = nn.Sequential(
    nn.Linear(in_features=INPUT_FEATURES, out_features=HIDDEN_SIZE),
    nn.ReLU(),
    nn.Linear(in_features=HIDDEN_SIZE, out_features=HIDDEN_SIZE),
    nn.ReLU(),
    nn.Linear(in_features=HIDDEN_SIZE, out_features=NUM_ACTIONS)
)

display(naive_q_network)

# 3. Simulate a batch from our Notebook 1 DataLoader
simulated_batch = {
    "state": torch.randn(4, 47, 4), # [Batch=4, Time=47, Features=4]
    "mask": torch.ones(4, 47, dtype=torch.bool),
    "original_lengths": torch.tensor([20, 30, 47, 17])
}

try:
    # We pass the dictionary directly into the model
    predictions = naive_q_network(simulated_batch)
except Exception as e:
    print(f"Error Type: {type(e).__name__}")
    print(f"Message: {e}")

Sequential(
  (0): Linear(in_features=4, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=2, bias=True)
)


--- Attempting Forward Pass ---
❌ CRASH DETECTED ❌
Error Type: TypeError
Message: linear(): argument 'input' (position 1) must be Tensor, not dict


nn.Sequential can work in simple DL solutions, and offers a very easy API to create model architectures. However once your problem or environment dictates more complex handling, you will have to abandon it.

**"What if I just pass `batch['state']` instead of the whole dictionary?"**
If you do that, the code will run, but the math will be completely poisoned. The network will calculate Q-values for all the fake padded zeros (e.g., calculating an action for timestep 46 of an episode that actually died at timestep 17). 

To fix this, we must throw away `nn.Sequential` and build a custom Object-Oriented class.

# nn.Module

To handle complex data structures (like dictionaries containing sequences and masks), we must subclass `torch.nn.Module`. 

In a custom PyTorch model, the architecture is strictly split into two phases:
1. **`__init__(self)` (The Hardware):** This is where we define our "Lego blocks". We instantiate the linear layers and store them in the class.
2. **`forward(self, batch)` (The Wiring):** This is where we define how the data physically flows through the blocks. We can unpack dictionaries, write `if` statements, and perform complex tensor math (like applying our padding mask).

In [7]:
class MaskedQNetwork(nn.Module):
    def __init__(self, input_features, hidden_size, num_actions):
        # 1. ALWAYS call the superclass init first, or PyTorch will crash
        super().__init__()
        
        # 2. Define the Lego Blocks
        self.layer1 = nn.Linear(input_features, hidden_size)
        self.relu1 = nn.ReLU()
        
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        
        self.output_layer = nn.Linear(hidden_size, num_actions)

    def forward(self, batch: dict) -> torch.Tensor:
        state = batch["state"]  # Shape: [Batch, Time, Features]
        mask = batch["mask"]    # Shape: [Batch, Time]
        
        x = self.layer1(state)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        
        # Raw Q-values (Shape: [Batch, Time, Actions])
        q_values = self.output_layer(x) 
        
        # 3. Apply the Padding Mask
        # We must add a dummy dimension to the mask so its shape [Batch, Time, 1]
        expanded_mask = mask.unsqueeze(-1)
        
        # Multiply! Any padded timestep is multiplied by 0 (False), erasing the fake math
        masked_q_values = q_values * expanded_mask
        
        return masked_q_values



In [9]:
lengths = torch.tensor([20, 30, 47, 17])
max_len = 47
batch_size = 4

# We use our broadcasting trick to generate a mathematically accurate dummy mask
fixed_mask = torch.arange(max_len).expand(batch_size, max_len) < lengths.unsqueeze(1)
simulated_batch = {
    "state": torch.randn(4, 47, 4), 
    "mask": fixed_mask,
    "original_lengths": lengths
}


production_q_net = MaskedQNetwork(input_features=4, hidden_size=64, num_actions=2)

# Pass the simulated dictionary from Step 4A
# Simulated shortest episode was length 17 (out of max 47)
safe_predictions = production_q_net(simulated_batch)

print(f"Final Prediction Shape: {safe_predictions.shape}")

# Let's inspect the math for the shortest episode (Index 3, Length 17)
print("\n--- Inspecting Padded Math (Shortest Episode) ---")
print("Timestep 16 (Real Data):")
print(safe_predictions[3, 16, :]) # Should show real, non-zero Q-values

print("\nTimestep 17 (Padded Zero):")
print(safe_predictions[3, 17, :]) # Should be strictly [0.0000, 0.0000]!

Final Prediction Shape: torch.Size([4, 47, 2])

--- Inspecting Padded Math (Shortest Episode) ---
Timestep 16 (Real Data):
tensor([-0.2733,  0.1852], grad_fn=<SelectBackward0>)

Timestep 17 (Padded Zero):
tensor([-0., 0.], grad_fn=<SelectBackward0>)


# nn vs F

In PyTorch, there are two ways to call activation functions and pooling operations:
* **`torch.nn` (Stateful):** Python classes that require instantiation in `__init__`.
* **`torch.nn.functional` (Stateless):** Pure mathematical functions called directly in `forward()`.

**The Golden Rule:** If a layer has trainable weights (like `Linear` or `Conv2d`), it MUST be an `nn.Module` in `__init__`. If an operation is purely mathematical and has no weights (like `ReLU`, `Softmax`, or reshaping), use `torch.nn.functional` in the `forward` pass to keep your architecture memory-efficient and clean.

In [ ]:
class ClutteredModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.relu1 = nn.ReLU() # Creates a useless object in memory
        self.layer2 = nn.Linear(20, 2)
        self.softmax = nn.Softmax(dim=1) # Another useless object
        
    def forward(self, x):
        x = self.relu1(self.layer1(x))
        return self.softmax(self.layer2(x))

class ProfessionalModel(nn.Module):
    def __init__(self):
        super().__init__()
        # ONLY store things that consume VRAM / have weights
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 2)
        
    def forward(self, x):
        # Use F for pure, weightless math
        x = F.relu(self.layer1(x))
        return F.softmax(self.layer2(x), dim=1)

The exceptions to this is what we had is you can't use F inside sequential, so if you want to group layers together in one sequential block, you will need to nn or create a custom Sequential Block.

The other exception is advanced but important, called Hooks. If we want to have a look at the data flowing through the network, you have to be using a stateful object to be able to do that. 

# Debugging

The most common error in PyTorch is a dimension mismatch. If your DataLoader outputs an image of size `[32, 3, 128, 128]` but your first linear layer expects exactly `1024` features, the model will instantly crash.

Instead of guessing where the math broke, we will add a `debug` flag to our model's initialization. When turned on, the `forward` pass will pause and print the exact mathematical shape of the tensor after every single layer.

In [16]:
class StandardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(in_features=20, out_features=64)
        self.layer2 = nn.Linear(in_features=64, out_features=2)
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = self.layer2(x)
        return x

# Create a dummy batch
bad_input = torch.randn(32,15) 
model = StandardModel()

print("--- Attempting Forward Pass ---")
try:
    output = model(bad_input)
except Exception as e:
    print(f"Error Type: {type(e).__name__}")
    print(f"Message: {e}")

--- Attempting Forward Pass ---
Error Type: RuntimeError
Message: mat1 and mat2 shapes cannot be multiplied (32x15 and 20x64)


In [17]:
class DebuggableModel(nn.Module):
    def __init__(self, debug_mode=False):
        super().__init__()
        self.debug_mode = debug_mode # Store the flag!
        
        # Notice we fixed the input feature size to 15
        self.layer1 = nn.Linear(15, 64) 
        self.layer2 = nn.Linear(64, 2)
        
    def forward(self, x):
        if self.debug_mode:
            print(f"[DEBUG] Input Shape: {x.shape}")
            
        x = self.layer1(x)
        
        if self.debug_mode:
            print(f"[DEBUG] After Layer 1: {x.shape}")
            
        x = F.relu(x)
        
        if self.debug_mode:
            print(f"[DEBUG] After ReLU: {x.shape}")
            
        x = self.layer2(x)
        
        if self.debug_mode:
            print(f"[DEBUG] Final Output Shape: {x.shape}")
            
        return x

print("\n--- Running with Debugger ON ---")
# Instantiate with debug_mode=True
safe_model = DebuggableModel(debug_mode=True)

# Pass the input (32, 15) through
# We will watch the tensor morph step-by-step
output = safe_model(bad_input) 


--- Running with Debugger ON ---
[DEBUG] Input Shape: torch.Size([32, 15])
[DEBUG] After Layer 1: torch.Size([32, 64])
[DEBUG] After ReLU: torch.Size([32, 64])
[DEBUG] Final Output Shape: torch.Size([32, 2])


Manually printing shapes inside the `forward` pass works, but it clutters your architecture code. Real research labs use two advanced methods to inspect network shapes from the *outside*.

1. **The External Standard (`torchinfo`):** A library that passes a dummy tensor through your network and generates a beautiful table of exact input/output shapes and parameter counts per layer.
2. **The Native Standard (PyTorch Hooks):** Using PyTorch's `register_forward_hook` to inject a spy function into every layer that automatically prints the shape, without modifying the `forward` pass at all.

In [21]:
class DebuggableModel(nn.Module):
    def __init__(self):
        super().__init__()        
        # Notice we fixed the input feature size to 15
        self.layer1 = nn.Linear(15, 64) 
        self.layer2 = nn.Linear(64, 2)
        
    def forward(self, x):
            
        x = self.layer1(x)
        
            
        x = F.relu(x)
            
        x = self.layer2(x)
            
        return x

In [26]:
from torchinfo import summary

print("--- Torchinfo Summary ---")
# We pass the model and the expected input size (Batch, Features)
# It automatically calculates exactly what happens inside!
summary(DebuggableModel(), input_size=(32, 15))

--- Torchinfo Summary ---


Layer (type:depth-idx)                   Output Shape              Param #
DebuggableModel                          [32, 2]                   --
├─Linear: 1-1                            [32, 64]                  1,024
├─Linear: 1-2                            [32, 2]                   130
Total params: 1,154
Trainable params: 1,154
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.04
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.00
Estimated Total Size (MB): 0.02

In [31]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

safe_model = DebuggableModel().to(device)

bad_input = torch.randn(32, 15).to(device) 

def shape_logger_hook(module, input_tensor, output_tensor):
    print(f"[{module.__class__.__name__}] Output Shape: {output_tensor.shape}")

hook_handles = []
for layer in safe_model.children():
    print(f"Registering hook for layer: {layer.__class__.__name__}")
    handle = layer.register_forward_hook(shape_logger_hook)
    hook_handles.append(handle)

_ = safe_model(bad_input)

# Clean up the hooks
for handle in hook_handles:
    handle.remove()

Using device: cuda
Registering hook for layer: Linear
Registering hook for layer: Linear
[Linear] Output Shape: torch.Size([32, 64])
[Linear] Output Shape: torch.Size([32, 2])


# Sub-Modules & `nn.Sequential`



PyTorch architectures are fractal. A `torch.nn.Module` can contain other `torch.nn.Module`s infinitely. 

When you have a block of operations where exactly *one* tensor goes in and *one* tensor comes out (like a standard feed-forward pipeline), you should group them using `nn.Sequential`. 

**The Best Practice:**
Instead of a massive list of individual layers, professional architectures are broken down into logical "Blocks" (e.g., a `feature_extractor` block and a `classification_head` block).

In [ ]:
class MessyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 32)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(32, 64)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(64, 2)

    def forward(self, x):
        # Very easy to accidentally forget a ReLU here
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        return self.output(x)


class CleanModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        self.classifier = nn.Linear(64, 2)

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)


pro_model = CleanModel()

print(pro_model)

CleanModel(
  (feature_extractor): Sequential(
    (0): Linear(in_features=10, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
  )
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


Note: You can't create F layers inside a sequential you will have to use nn or create a custom sequential instead

# Custom Weight Initialization

When you create a layer like nn.Linear(64, 64), PyTorch fills the weight matrix with random numbers. If those random numbers are slightly too small, your data shrinks as it passes through the layers until it becomes 0.0 (Vanishing). If they are slightly too large, your data multiplies until it becomes NaN or infinity (Exploding).

The Focus: We will look at why the default PyTorch initialization is sometimes insufficient for very deep networks, especially those using ReLU activations.

Instead of manually generating random tensors, PyTorch provides a dedicated library called torch.nn.init containing the exact mathematical formulas from famous research papers.

We will focus on Kaiming (He) Initialization, which is the absolute industry standard mathematically designed to keep the variance of data perfectly stable when using ReLU activations.

We just built a beautiful, nested architecture in Step 2D using nn.Sequential. If we want to change the weights, how do we reach inside all those nested sub-modules without writing a massive, ugly for loop?

We will write a standalone Python function that applies the Kaiming math, and then use the model.apply(function) method. This acts like a spider that recursively crawls through every single nested block of your architecture and overwrites the weights.